# Databricks Workflow
- Trigger types:
- - Manual
- - Scheduled (Cron)
- - API
- - File arrival
- - Delta table update
- - Continous Streaming

DAG stands for Directed Acyclic Graph, a data structure that consists of vertices (or nodes) connected by edges that have a direction associated with them, forming a graph with no directed cycles. This means there are no sequences of edges where you can start from one node and loop back to it following the directed edges. DAGs are used in various applications such as scheduling, data processing, and as an underlying structure in some cryptocurrencies.

# DAG patterns
- Sequence - ETL, Bronze, Silver, Gold
- Funnel - Multiple data source, Data collection
- Fan-out - Single data source, ingestion and distribution 

### Job can be cretated by: UI, CLI, Terraform, Jobs API

# Databricks Artifact Bundle aka DAB - is collection of Databricks artifacts or bundles.

```
databricks bundle deploy -e 'dev'
databricks bundle run pipeline --refresh-all -e 'dev'
```
Workflow params
```
dbutils.jobs.taskValues.set(key='bad_records', value=5) # set in notebook
{{tasks.TaskName.values.bad_records}} # set in job if-else conditions
```

# Data pipeline with Delta Live Tables

Medallion arch - is a data management framework designed to handle the complete lifecycle of data within the Databricks platform, from ingestion to transformation, and serving. It organizes data into three layers known as Bronze, Silver, and Gold, each representing different stages of refinement, reliability, and readiness for consumption.

Bronze (Raw Layer): This is the initial landing layer where raw data is ingested directly from various sources. The data in this layer is kept in its original form, often with minimal processing.

Silver (Clean Layer): In the Silver layer, the data undergoes further cleaning, enrichment, and transformation. It becomes more structured and reliable, suitable for deeper analysis and queries by data teams.

Gold (Business Layer): The Gold layer contains the most refined and business-ready datasets, often involving aggregations, metrics calculations, or summaries that directly support decision-making processes and business intelligence activities.


Bronze (Raw Layer) -> Silver (Clean Layer) -> Gold (Business Layer)

# Live table is Materialized view for the lakehouse


```
CREATE LIVE TABLE report AS SELECT SUM(profit) FROM prod.sales; -- defined by SQL, created and updated by pipeline

CREATE STREAM TABLE mystream AS SELECT * FROM STREAM(mytable); 
```

# DLT - Declared Live Table dependency

```
CREATE LIVE TABLE AS SELECT * FROM LIVE.report; -- it handles parallelizm
```

# Expectations

```
... CONSTRAINT valid_timestamp EXPECT(timestamp > '2020-01-01') ON VIOLATION DROP
```

# Create streaming table from files

```
CREATE STREAMING LIVE TABLE data AS SELECT * FROM cloud_files("path/to/files", "json");
```

# Apply changes to CDC

```
APPLY CHANGES INTO LIVE.events
FROM STREAM (LIVE.events_updates)
KEYS(id)
SEQUENCE BY ts;
```

##### This code snippet appears to apply changes from an update stream LIVE.events_updates into a live database or dataset LIVE.events. Changes are matched based on the id field (specified by KEYS(id)) and it's important to apply updates in the order indicated by the ts (timestamp) field (as specified by SEQUENCE BY ts). This ensures that the latest updates are applied in the correct order to the live dataset.

#### Using autoloader
- cloud_files() - enables auto-loader for incrementaly loads data from the source and track data changing/ files incomming

```
CREATE OR REFRESH STREAMING TABLE order_bronze
AS SELECT CURRENT_TIMESTAMP() as processing_time, *
FROM cloud_files('path/to/data', 'json', map('cloud_files.inferColumnTypes', 'true'));
```

```
CREATE OR REFRESH STREAMING TABLE order_silver
(CONSTRAINT valid_date EXPECT (order_timestamp "2021-01-1") ON VIOLATION FAIL UPDATE)                   -- If this constraint is violated, the update will fail.
COMMENT "Append only orders with valid timestamps"
TBLPROPERTIES ("quality"="silver")                                                                       --  data quality of the table.
AS SELECT timestamp(order_timestamp) AS order_timestamp, SELECT * EXCEPT(order_timestamp, _rescued_data) -- Select all columns except provided from the source.
FROM STREAM(LIVE.order_bronze);
```

Live table is another meaning of Materialized View

### Declare live table by python code 

```
import dlt

@dlt.table
def order_bronze():
  return (
    spark.readStream.format('cloudFiles')
    .option('cloudFiles.format', 'json')
    .option('cloudFiles.inferColumnTypes', 'true')
    .load('path/to/data')
    .select(F.current_timestamp().alias('processing_time'), "*")
  )

# Add comments, properties and constraints
@dlt.table(
  comment="Append only orders with valid records",
  table_properties={"quality":"silver"}
)
@dlt.expect_or_fail("valid_date", F.col("order_timestamp") > "2021-01-01")
def order_silver():
  return (
    dlt.read_stream('order_bronze')
    .select('processing_time', 'customer_id', 'notifications', 'order_id',
    F.col('order_timestamp').cast('timestamp').alias('order_timestamp'))
  )
```

# Create LIVE VIEW 

```
CREATE LIVE VIEW subscribed_order 
AS SELECT a.customer_id, a.order_id, b.email
FROM LIVE.order_silver AS a 
INNER JOIN LIVE.customer_silver AS b ON a.customer_id = b.customr_id
WHERE notifications = 'Y';
```